# Lesson 3 - Motors

**Safety: lift the wheels off the ground before running any cell in this notebook.**

The two drive motors are controlled through a PCA9685 chip at I2C address `0x60`.
`Robot.set_motors(left, right)` takes a value from -1.0 to 1.0 for each wheel.

In [ ]:
import sys
sys.path.insert(0, "..")
from jbot import Robot

robot = Robot()

## Direction buttons

Each button drives for a short moment. Use the slider to change the speed and the
red **STOP** button to halt immediately.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

speed = widgets.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.05, description="speed")

def make(name, fn):
    b = widgets.Button(description=name)
    b.on_click(lambda _: fn(speed.value))
    return b

up = make("forward", robot.forward)
down = make("backward", robot.backward)
left = make("left", robot.left_turn)
right = make("right", robot.right_turn)
stop = widgets.Button(description="STOP", button_style="danger")
stop.on_click(lambda _: robot.stop())

display(speed)
display(widgets.HBox([left, up, right]))
display(widgets.HBox([down, stop]))

## Tank-style control with two sliders

Drag each slider to set the left and right wheel speed directly. The robot updates
live as you move them.

In [ ]:
left_s = widgets.FloatSlider(value=0, min=-1, max=1, step=0.05, description="left", orientation="vertical")
right_s = widgets.FloatSlider(value=0, min=-1, max=1, step=0.05, description="right", orientation="vertical")

def update(_):
    robot.set_motors(left_s.value, right_s.value)

left_s.observe(update, "value")
right_s.observe(update, "value")

reset = widgets.Button(description="STOP", button_style="danger")
def on_reset(_):
    left_s.value = 0
    right_s.value = 0
reset.on_click(on_reset)

display(widgets.HBox([left_s, right_s, reset]))

## Visualize the motion vector

Forward speed is the average of the two wheels; turn rate is their difference. The
arrow shows the resulting motion command.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

def plot_motion(l, r):
    forward = (l + r) / 2.0
    turn = (r - l) / 2.0
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.arrow(0, 0, turn, forward, head_width=0.08, color="tab:blue", length_includes_head=True)
    ax.axhline(0, color="gray", lw=0.5)
    ax.axvline(0, color="gray", lw=0.5)
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_aspect("equal")
    ax.set_title("forward=%.2f  turn=%.2f" % (forward, turn))
    plt.show()

plot_motion(left_s.value, right_s.value)

In [ ]:
robot.stop()